In [42]:
import pandas as pd
import numpy as np

sol = pd.read_csv("FinAcceso_solicitudes_credito.csv")
mer = pd.read_csv("FinAcceso_mercado_potencial.csv")


### Punto 1
Supuestos implícitos

Que más créditos aprobados significa más demanda. Pero las aprobaciones dependen del puntaje y los criterios de la entidad, no solo de cuánta gente necesita el servicio.
Que la asesoría rinde más donde ya se aprueba más. Podría ser al revés: donde se aprueba poco es donde más falta hace.

El archivo de solicitudes describe solo a quienes pidieron un crédito. El de mercado potencial es una estimación de los hogares que podrían estar interesados en cada zona.

Quedan invisibles los hogares que nunca solicitaron, por ejemplo por falta de información, dificultad para acceder al canal o porque creen que los van a rechazar.

¿Las zonas con menos aprobaciones tienen realmente menos hogares interesados, o simplemente solicitan menos y se les aprueba menos en proporción a su mercado?

In [43]:
sol["zona"].value_counts(normalize = True).reset_index()

,zona,proportion
0,Norte,0.359848
1,Centro,0.299242
2,Sur,0.240909
3,Periferia,0.100000


| Zona | Hogares potenciales | Porcentaje |
|---|---|---|
| Norte | 1400 | 14 % |
| Centro | 2200 | 22 % |
| Sur | 3700 | 37 % |
| Periferia | 2700 | 27 % |
| **Total** | **10000** | **100 %** |

# Punto 2

In [44]:
tablaz = sol.merge(mer, on = "zona", how = "inner")
tablaz

,id_solicitud,fecha_solicitud,zona,canal,ingreso_mensual_cop,historial_crediticio,puntaje_riesgo,carga_financiera,monto_solicitado_cop,decision,hogares_potenciales,descripcion_fuente
0,FA-00003,2026-07-01,Sur,Digital,2684000,Limitado,598,0.176,5800000,Rechazada,3700,Estimación administrativa simulada
1,FA-00024,2026-07-01,Sur,Digital,1671000,Amplio,546,0.622,980000,Rechazada,3700,Estimación administrativa simulada
2,FA-00035,2026-07-01,Centro,Digital,3984000,Amplio,753,0.590,2360000,Aprobada,2200,Estimación administrativa simulada
3,FA-00460,2026-07-01,Norte,Digital,3362000,Amplio,605,0.620,4990000,Aprobada,1400,Estimación administrativa simulada
4,FA-00661,2026-07-01,Periferia,Digital,1715000,Limitado,566,0.437,1680000,Rechazada,2700,Estimación administrativa simulada
...,...,...,...,...,...,...,...,...,...,...,...,...
1315,FA-01115,2026-09-30,Centro,Presencial,3564000,Amplio,646,0.614,1830000,Rechazada,2200,Estimación administrativa simulada
1316,FA-01137,2026-09-30,Centro,Digital,1912000,Amplio,679,0.631,4380000,Aprobada,2200,Estimación administrativa simulada
1317,FA-01172,2026-09-30,Sur,Digital,1887000,Limitado,577,0.374,3760000,Rechazada,3700,Estimación administrativa simulada
1318,FA-01194,2026-09-30,Centro,Digital,2133000,Amplio,573,0.390,5890000,Rechazada,2200,Estimación administrativa simulada


In [45]:


tablazo = tablaz.groupby("zona").agg(
    solicitudes_recibidas = ("zona", "value_counts"),
    hogares_potenciales = ("hogares_potenciales", "unique"),
)
tablazo["porcentaje del total de solicitudes"] = tablaz["zona"].value_counts(normalize = True) * 100
tablazo["solicitudes por cada 1.000 hogares potencial"] = (tablaz["zona"].value_counts() * 1000) / tablaz["hogares_potenciales"].unique()[1]
tablazo

,solicitudes_recibidas,hogares_potenciales,porcentaje del total de solicitudes,solicitudes por cada 1.000 hogares potencial
zona,,,,
Centro,395,[2200],29.924242,179.545455
Norte,475,[1400],35.984848,215.909091
Periferia,132,[2700],10.000000,60.000000
Sur,318,[3700],24.090909,144.545455


#### no necesariamente ya que depende de la cantidad de hogares potenciales ya que puden haber pocas solicitudes y aun asi que sea una gran parte de la zona la que se encuentra interesada en el servicio
#### Por lo tanto es importante entrar a revisar la cantidad de solicitudes por cada 1000 hogares ya que puede dar una mejor percepcion de lo que realmente se esta presentando y de esta manera mandar mayor asistencia a los que presenten mayor solicitudes cada 1000 hogares

# Punto 3

In [46]:
sol["decision"].value_counts().reset_index()

,decision,count
0,Rechazada,662
1,Aprobada,658


In [47]:
sol["aprobado"] = (sol["decision"] == "Aprobada").astype(int)
sol["aprobado"].mean() * 100

np.float64(49.848484848484844)

La tasa global de aprobación es 49,8 %

In [48]:
t3 = sol.groupby("zona").agg(
    solicitudes=("aprobado", "count"),
    aprobaciones=("aprobado", "sum"),
    tasa_aprob=("aprobado", "mean")
)
t3["tasa_aprob"] = (t3["tasa_aprob"] * 100)
t3.sort_values("aprobaciones", ascending=False)

,solicitudes,aprobaciones,tasa_aprob
zona,,,
Norte,475,295,62.105263
Centro,395,193,48.860759
Sur,318,129,40.566038
Periferia,132,41,31.060606


-El orden de las zonas no cambia entre números absolutos y tasas

-Sería engañoso interpretar que Norte tiene más demanda o que allí la asesoría rinde más. Ni las aprobaciones ni la tasa miden necesidad la tasa refleja quienes lo solicitan.

-Antes de comparar tasas entre zonas conviene revisar el puntaje de riesgo, porque si los solicitantes de Periferia tienen puntajes más bajos, su menor tasa se explicaría por el puntaje y no por la zona, y ese puntaje heredado podría reflejar desigualdades previas.

# Punto 4

In [49]:
sol.groupby("zona")["puntaje_riesgo"].median()

zona
Centro       659.0
Norte        670.0
Periferia    594.0
Sur          614.5
Name: puntaje_riesgo, dtype: float64

In [50]:
sol["historial_crediticio"].value_counts()

historial_crediticio
Amplio      824
Limitado    496
Name: count, dtype: int64

In [51]:
def rango_puntaje(p):
    if p < 610:
        return "<610"
    elif p < 690:
        return "610-689"
    else:
        return "690+"

sol["rango_puntaje"] = sol["puntaje_riesgo"].apply(rango_puntaje)
sol["rango_puntaje"].value_counts()

rango_puntaje
610-689    490
<610       448
690+       382
Name: count, dtype: int64

In [52]:
t4 = sol.groupby(["rango_puntaje", "historial_crediticio"]).agg(
    solicitudes=("decision", "count"),
    pct_aprob=("decision", lambda x: (x == "Aprobada").mean() * 100)
)
t4

solicitudes  pct_aprob
rango_puntaje historial_crediticio                        
610-689       Amplio                        319  58.307210
              Limitado                      171  44.444444
690+          Amplio                        318  82.704403
              Limitado                       64  73.437500
<610          Amplio                        187  24.598930
              Limitado                      261  15.325670

En todos los intervalos de puntaje, las personas con historial amplio se aprueban más que las de historial limitado, y la mayoría de quienes tienen historial limitado están en el rango de puntaje más bajo.

No se puede concluir.Faltaría revisar ingreso, carga financiera, monto, canal y zona, que podrían explicar parte de la brecha. Tampoco sabemos cómo se construyó el puntaje heredado de puntaje de riesgo, si ya incluye el historial, el limitado se penaliza dos veces. Y no tenemos datos de mora ni de quienes nunca solicitaron, así que no sabemos si el historial limitado realmente implica más riesgo.

# Punto 5

#### si se elimina la asesoria se recorta en las zonas con menos aprobaciones perderiamos y dejariamos de tener encuenta a las zonas con mayor cantidad de hogares potenciales lo cual representaria a futuron un mercado no explorado y en cuestion de impactos inmediatos no se veria muy afectado ya que no se cuenta con muchas aprobaciones lo que podria llegar a pasar es que las aprobaciones tiendan a seguir bajando 

#### teniendo en cuenta que se perderia de vista zonas con bastante potencial de crecimiento y teniendo encuenta que las personas con un historial limitado suelen ser menos aprobadas que las demas y presentamos un bajo indice de aporbacion en esta zona las personas y grupos que mas se verian afectados serian las personas de historial limitado sobre todo de la zona periferia

#### Es importante primero conocer cuantos hogares son los que presentan solicitudes para esto se recomendaria crear otra tabla que sea de hogares donde podemos poner variables como estrato, personas viviendo en el sitio entre muchas otras, para conocer que familias son mas propensas a tener creditos aprobados. 
#### La segunda recomendacion seria ver como la implementacion de la decision afecta al grupo mas vulnerado de la zona de la periferia con historial limitado ya que desde ahi se puden tomar mas y mejores desiciones con el fin de aumentar la tasa de aprobacion teniendo como un posible objetivo el 70% de aprobaciones con hogares de todas las zonas evitando enfocarnos solo en una parte de los hogares de la ciudad 

# Punto 6

#### Principalmente en este caso lo mejor es modificar la implementación de la decisión y empezar por tomar más datos relevantes como la composición del hogar para conocer cuáles son más propensos a tener una aprobación y de esta manera enfocar los recursos a hogares con una tasa alta de aprobación ya que de incluimos todas las zonas de la ciudad. Por otro lado, la zona norte tiene la mayor tasa de aprobación y también la que menos hogares potenciales presenta y la zona norte tiene un 35% de las solicitudes totales, cuando en la población responde a un 14% lo que genera un sesgo hacia esta zona, además el hecho de no saber cuántos hogares realmente pasaron las solicitudes no podemos saber cuántos solicitudes hay realmente cada 1000 hogares siendo una variable importante y que nos limita a tomar mejores decisiones.

#### Es tentador decir que las personas de la zona de la periferia 